# Conciliação de Benefícios
**Instituto Atlântico · Departamento Pessoal**

---

Este notebook realiza a conciliação mensal de benefícios: confronta os valores cobrados pelos fornecedores com os dados de referência do DP e gera um relatório de divergências por colaborador.

**O que você vai precisar ter em mãos:**
- As faturas dos fornecedores (arquivos Excel enviados mensalmente por cada fornecedor)
- A referência interna do DP (`descontos_proventos_beneficios_MMAAAA.xlsx`)

**O que este notebook entrega:**
- Resumo consolidado por fornecedor
- Detalhe das divergências por colaborador
- Export opcional em Excel para registro

---

### Como usar

| Etapa | O que fazer |
|---|---|
| **Célula 1** | Preparar o ambiente — execute uma vez ao abrir o notebook |
| **Célula 2** | Carregar o sistema — execute uma vez por sessão |
| **Célula 3** | Selecionar os arquivos e confirmar |
| **Célula 4** | Visualizar o resultado da conciliação |

In [ ]:
#@title Célula 1 · Preparação do ambiente { display-mode: "form" }
#@markdown Execute uma única vez ao abrir o notebook.

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "pandas"])

import io
import pandas as pd
import openpyxl
from IPython.display import display, HTML
from google.colab import files
import ipywidgets as widgets

# Estado global compartilhado entre células
estado = {
    "uploaded":           {},   # {nome_arquivo: bytes}
    "faturas":            {},   # {chave_fornecedor: nome_arquivo}
    "arquivo_referencia": None,
}

print("✅ Ambiente pronto.")

In [ ]:
#@title Célula 2 · Carregamento do sistema { display-mode: "form" }
#@markdown Execute uma única vez por sessão.
# logic.py · sync: AAAA-MM-DD

TOLERANCIA = 0.05


# ════════════════════════════════════════════════════════════════════════
# PARSER: Bradesco Dental
# ════════════════════════════════════════════════════════════════════════

def parse_fatura_bradesco_dental(conteudo):
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["bradesco"]

    header_row = None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if any(str(v).strip().lower() == "matricula titular" for v in row if v):
            header_row = i
            break
    if header_row is None:
        raise ValueError("Coluna 'Matricula Titular' não encontrada na aba 'bradesco'.")

    headers = list(ws.iter_rows(values_only=True))[header_row]
    idx = {}
    for i, h in enumerate(headers):
        h_norm = str(h).strip().lower() if h else ""
        if h_norm == "matricula titular":      idx["mat"]     = i
        elif h_norm == "desconto colaborador": idx["desconto"] = i
        elif h_norm in ["x", "custo"] and "custo" not in idx:
                                               idx["custo"]   = i

    registros = []
    for row in list(ws.iter_rows(values_only=True))[header_row + 2:]:
        mat   = row[idx["mat"]]      if "mat"     in idx else None
        desc  = row[idx["desconto"]] if "desconto" in idx else None
        custo = row[idx["custo"]]    if "custo"   in idx else None
        if mat and isinstance(mat, (int, float)) and desc:
            registros.append({
                "matricula":       int(mat),
                "desconto_fatura": float(desc),
                "custo_fatura":    float(custo) if custo else 0.0,
            })

    df = pd.DataFrame(registros)
    return df.groupby("matricula").agg(
        desconto_fatura=("desconto_fatura", "sum"),
        custo_fatura=("custo_fatura", "sum"),
        qtd_vidas=("matricula", "count"),
    ).reset_index()


# ════════════════════════════════════════════════════════════════════════
# PARSER: Unimed
# ════════════════════════════════════════════════════════════════════════

def parse_fatura_unimed(conteudo):
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["unimed"]

    header_row = None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if any(str(v).strip().lower() == "matricula titular" for v in row if v):
            header_row = i
            break
    if header_row is None:
        raise ValueError("Coluna 'Matricula Titular' não encontrada na aba 'unimed'.")

    headers = list(ws.iter_rows(values_only=True))[header_row]
    idx = {}
    for i, h in enumerate(headers):
        h_norm = str(h).strip().lower() if h else ""
        if h_norm == "matricula titular":    idx["mat"]     = i
        elif h_norm == "desconto colaborador": idx["desconto"] = i
        elif h_norm == "custo atlantico":      idx["custo"]   = i

    missing = [k for k in ("mat", "desconto", "custo") if k not in idx]
    if missing:
        raise ValueError(f"Colunas não encontradas na aba 'unimed': {missing}")

    registros = []
    for row in list(ws.iter_rows(values_only=True))[header_row + 1:]:
        mat   = row[idx["mat"]]
        desc  = row[idx["desconto"]]
        custo = row[idx["custo"]]
        if mat and isinstance(mat, (int, float)) and isinstance(desc, (int, float)):
            registros.append({
                "matricula":       int(mat),
                "desconto_fatura": float(desc),
                "custo_fatura":    float(custo) if isinstance(custo, (int, float)) else 0.0,
            })

    df = pd.DataFrame(registros)
    return df.groupby("matricula").agg(
        desconto_fatura=("desconto_fatura", "sum"),
        custo_fatura=("custo_fatura", "sum"),
        qtd_vidas=("matricula", "count"),
    ).reset_index()


# ════════════════════════════════════════════════════════════════════════
# PARSER: Referência interna do DP
# ════════════════════════════════════════════════════════════════════════

def parse_referencia_interna(conteudo, fornecedor_label):
    wb = openpyxl.load_workbook(io.BytesIO(conteudo), data_only=True)
    ws = wb["total"]
    rows = list(ws.iter_rows(values_only=True))

    idx_mat = idx_nome = idx_fatura = idx_desconto = idx_custo = None
    idx_bd_start = header1_row = header2_row = None

    for i, row in enumerate(rows):
        vals = [str(v).strip().lower() if v else "" for v in row]
        if "mat" in vals and "nome" in vals:
            header1_row = i
            idx_mat  = next(j for j, v in enumerate(vals) if v == "mat")
            idx_nome = next(j for j, v in enumerate(vals) if v == "nome")
            for j, v in enumerate(row):
                if v and fornecedor_label.lower() in str(v).lower():
                    idx_bd_start = j
                    header2_row  = i + 1
                    break
            break

    if header2_row is None or idx_bd_start is None:
        raise ValueError(
            f"Cabeçalho do fornecedor '{fornecedor_label}' não encontrado "
            f"na aba 'total'."
        )

    subheaders = rows[header2_row]
    for j in range(idx_bd_start, min(idx_bd_start + 10, len(subheaders))):
        v = subheaders[j]
        if v is None:
            continue
        v_low = str(v).strip().lower()
        if "fatura" in v_low and idx_fatura is None:
            idx_fatura = j
        elif "desconto" in v_low and idx_fatura is not None and idx_desconto is None:
            idx_desconto = j
        elif "custo" in v_low and idx_desconto is not None and idx_custo is None:
            idx_custo = j
            break

    if None in (idx_fatura, idx_desconto, idx_custo):
        raise ValueError(
            f"Subcolunas Fatura/Desconto/Custo não encontradas para '{fornecedor_label}'."
        )

    registros = []
    for row in rows[header2_row + 1:]:
        mat   = row[idx_mat]      if idx_mat      is not None else None
        nome  = row[idx_nome]     if idx_nome     is not None else None
        fat   = row[idx_fatura]   if idx_fatura   is not None else None
        desc  = row[idx_desconto] if idx_desconto is not None else None
        custo = row[idx_custo]    if idx_custo    is not None else None
        if mat and isinstance(mat, (int, float)):
            registros.append({
                "matricula":         int(mat),
                "nome":              nome,
                "desconto_esperado": float(desc)  if isinstance(desc,  (int, float)) else 0.0,
                "fatura_esperada":   float(fat)   if isinstance(fat,   (int, float)) else 0.0,
                "custo_esperado":    float(custo) if isinstance(custo, (int, float)) else 0.0,
            })
    return pd.DataFrame(registros)


def conciliar(df_fatura, df_referencia):
    df = df_referencia.merge(df_fatura, on="matricula", how="outer")
    df["desconto_fatura"]   = df["desconto_fatura"].fillna(0.0)
    df["desconto_esperado"] = df["desconto_esperado"].fillna(0.0)
    df["diferenca"]         = (df["desconto_fatura"] - df["desconto_esperado"]).round(4)

    def _status(row):
        if abs(row["diferenca"]) <= TOLERANCIA:  return "✅ OK"
        elif row["desconto_esperado"] == 0:       return "👻 Só na fatura"
        elif row["desconto_fatura"]   == 0:       return "🔍 Só na referência"
        else:                                     return "💰 Divergência de valor"

    df["status"] = df.apply(_status, axis=1)
    return df


PARSERS = {
    ("bradesco", "dental"): parse_fatura_bradesco_dental,
    ("unimed",):            parse_fatura_unimed,
}

FORNECEDOR_LABELS = {
    ("bradesco", "dental"): "bradesco dental",
    ("unimed",):            "unimed",
}


def identificar_fornecedor(nome_arquivo):
    nome_lower = nome_arquivo.lower()
    for chave, fn in PARSERS.items():
        if all(k in nome_lower for k in chave):
            return chave, fn
    return None, None


REFERENCIA_KEYWORDS = ("descontos", "proventos")

fornecedores_disponiveis = [
    " ".join(chave).title() for chave in PARSERS.keys()
]

print("✅ Sistema carregado.")
print(f"   Fornecedores disponíveis: {', '.join(fornecedores_disponiveis)}")

In [ ]:
#@title Célula 3 · Seleção de arquivos { display-mode: "form" }
#@markdown Selecione a referência interna e as faturas dos fornecedores.
#@markdown Você pode selecionar múltiplas faturas de uma vez.
#@markdown Clique em **Confirmar** quando todos os arquivos estiverem carregados.

from IPython.display import clear_output

SLOT_VAZIO = "<span style='color:#bbb;'>— aguardando arquivo</span>"
SLOT_OK    = "<span style='color:#2e7d32; font-weight:600;'>✅ {nome}</span>"
SLOT_ERRO  = "<span style='color:#c62828;'>⚠️ {nome} — {motivo}</span>"

btn_ref = widgets.Button(
    description="📋 Referência interna (DP)",
    button_style="info",
    layout=widgets.Layout(width="260px", height="44px")
)
btn_fat = widgets.Button(
    description="🧾 Faturas dos fornecedores",
    button_style="info",
    layout=widgets.Layout(width="260px", height="44px")
)
btn_confirmar = widgets.Button(
    description="✔ Confirmar",
    button_style="success",
    layout=widgets.Layout(width="140px", height="36px"),
    disabled=True
)
btn_limpar = widgets.Button(
    description="✖ Limpar",
    button_style="warning",
    layout=widgets.Layout(width="100px", height="36px")
)

out_log    = widgets.Output()
out_status = widgets.Output()


def _render_status():
    ref_nome = estado["arquivo_referencia"]
    faturas  = estado["faturas"]

    slot_ref = SLOT_OK.format(nome=ref_nome) if ref_nome else SLOT_VAZIO

    if faturas:
        slots_fat = "".join(
            f"<div>{SLOT_OK.format(nome=f'{nome} ({chr(32).join(chave).title()})')}</div>"
            for chave, nome in faturas.items()
        )
    else:
        slots_fat = SLOT_VAZIO

    pronto = bool(ref_nome and faturas)
    btn_confirmar.disabled = not pronto

    rodape = (
        "<div style='margin-top:14px; padding:10px 14px; background:#e8f5e9; "
        "border-left:3px solid #2e7d32; border-radius:4px; font-size:13px; color:#1b5e20;'>"
        f"✅ <strong>{len(faturas)} fatura(s) carregada(s).</strong> Clique em <strong>Confirmar</strong> para continuar."
        "</div>"
        if pronto else ""
    )

    html = f"""
    <div style='font-family:sans-serif; font-size:14px; padding:12px 0;'>
      <table style='border-collapse:collapse; width:100%;'>
        <tr>
          <td style='padding:6px 16px 6px 0; color:#555; white-space:nowrap; vertical-align:top;'>
            📋 Referência interna
          </td>
          <td style='padding:6px 0;'>{slot_ref}</td>
        </tr>
        <tr>
          <td style='padding:6px 16px 6px 0; color:#555; white-space:nowrap; vertical-align:top;'>
            🧾 Faturas
          </td>
          <td style='padding:6px 0;'>{slots_fat}</td>
        </tr>
      </table>
      {rodape}
    </div>
    """
    with out_status:
        clear_output(wait=True)
        display(HTML(html))


def _carregar_referencia(b):
    with out_log:
        novos = files.upload()
    with out_log:
        clear_output(wait=True)
    for nome, conteudo in novos.items():
        if all(k in nome.lower() for k in REFERENCIA_KEYWORDS):
            estado["uploaded"][nome]     = conteudo
            estado["arquivo_referencia"] = nome
        else:
            with out_status:
                clear_output(wait=True)
                display(HTML(
                    f"<div style='font-family:sans-serif; font-size:14px; padding:10px 14px; "
                    f"background:#fff3e0; border-left:3px solid #f57c00; border-radius:4px;'>"
                    f"⚠️ <strong>{nome}</strong> não reconhecido como referência interna.<br>"
                    f"<span style='color:#777; font-size:13px;'>O nome deve conter "
                    f"<em>descontos</em> e <em>proventos</em>.</span></div>"
                ))
            return
    _render_status()


def _carregar_faturas(b):
    with out_log:
        novos = files.upload()
    with out_log:
        clear_output(wait=True)

    nao_reconhecidos = []
    for nome, conteudo in novos.items():
        chave, _ = identificar_fornecedor(nome)
        if chave:
            estado["uploaded"][nome]   = conteudo
            estado["faturas"][chave]   = nome
        else:
            nao_reconhecidos.append(nome)

    if nao_reconhecidos:
        disponiveis = ", ".join(" ".join(c).title() for c in PARSERS.keys())
        avisos = "".join(f"<li>{n}</li>" for n in nao_reconhecidos)
        with out_status:
            clear_output(wait=True)
            display(HTML(
                f"<div style='font-family:sans-serif; font-size:14px; padding:10px 14px; "
                f"background:#fff3e0; border-left:3px solid #f57c00; border-radius:4px;'>"
                f"⚠️ Arquivo(s) não reconhecido(s):<ul style='margin:6px 0 4px 16px;'>{avisos}</ul>"
                f"<span style='color:#777; font-size:13px;'>Fornecedores disponíveis: <em>{disponiveis}</em>.</span></div>"
            ))

    _render_status()


def _limpar(b):
    estado["uploaded"].clear()
    estado["faturas"].clear()
    estado["arquivo_referencia"] = None
    btn_confirmar.disabled = True
    with out_status:
        clear_output(wait=True)
    _render_status()


def _confirmar(b):
    btn_confirmar.disabled = True
    with out_status:
        clear_output(wait=True)
        display(HTML(
            "<div style='font-family:sans-serif; font-size:14px; padding:10px 14px; "
            "background:#e8f5e9; border-left:3px solid #2e7d32; border-radius:4px; color:#1b5e20;'>"
            "✅ <strong>Confirmado.</strong> Execute a <strong>Célula 4</strong> para ver o resultado."
            "</div>"
        ))


btn_ref.on_click(_carregar_referencia)
btn_fat.on_click(_carregar_faturas)
btn_limpar.on_click(_limpar)
btn_confirmar.on_click(_confirmar)

display(widgets.VBox([
    widgets.HTML("<div style='font-family:sans-serif; font-size:13px; color:#555; margin-bottom:6px;'>1. Selecione os arquivos</div>"),
    widgets.HBox([btn_ref, btn_fat], layout=widgets.Layout(gap="12px")),
    widgets.HTML("<div style='font-family:sans-serif; font-size:13px; color:#555; margin:12px 0 6px 0;'>2. Confirme quando todos estiverem carregados</div>"),
    widgets.HBox([btn_limpar, btn_confirmar], layout=widgets.Layout(gap="8px")),
    out_log,
    out_status,
]))
_render_status()

In [ ]:
#@title Célula 4 · Resultado da conciliação { display-mode: "form" }
#@markdown Execute após confirmar os arquivos na Célula 3.
#@markdown
#@markdown **Quer salvar o resultado em Excel?** Marque a opção abaixo *antes* de executar.
exportar_excel = False #@param {type:"boolean"}

from datetime import datetime as dt
from IPython.display import clear_output

if not estado["faturas"] or not estado["arquivo_referencia"]:
    display(HTML("""
    <div style="font-family:sans-serif; padding:12px; background:#fff3e0;
                border-left:4px solid #f57c00; border-radius:4px;">
      ⚠️ Nenhum arquivo confirmado. Execute a <strong>Célula 3</strong> primeiro.
    </div>
    """))
else:
    conteudo_ref = estado["uploaded"][estado["arquivo_referencia"]]
    resultados   = {}  # {chave: df_result}

    # ── Processar todos os fornecedores ───────────────────────────────────────
    erros = []
    for chave, nome_arquivo in estado["faturas"].items():
        try:
            _, fn_parser     = identificar_fornecedor(nome_arquivo)
            fornecedor_label = FORNECEDOR_LABELS[chave]
            df_fatura        = fn_parser(estado["uploaded"][nome_arquivo])
            df_referencia    = parse_referencia_interna(conteudo_ref, fornecedor_label)
            resultados[chave] = conciliar(df_fatura, df_referencia)
        except Exception as e:
            erros.append((" ".join(chave).title(), str(e)))

    if erros:
        erros_html = "".join(
            f"<li><strong>{f}</strong>: {msg}</li>" for f, msg in erros
        )
        display(HTML(
            f"<div style='font-family:sans-serif; padding:10px 14px; background:#fff3e0; "
            f"border-left:4px solid #f57c00; border-radius:4px; margin-bottom:12px;'>"
            f"⚠️ Erro ao processar:<ul style='margin:6px 0 0 16px;'>{erros_html}</ul></div>"
        ))

    if not resultados:
        display(HTML("<p style='font-family:sans-serif; color:#c62828;'>Nenhum fornecedor processado com sucesso.</p>"))
    else:
        hoje = dt.today().strftime("%d/%m/%Y")

        # ── Resumo consolidado ────────────────────────────────────────────────
        linhas_resumo = []
        for chave, df in resultados.items():
            label      = " ".join(chave).title()
            total      = len(df[df["desconto_esperado"] > 0])
            qtd_ok     = (df["status"] == "✅ OK").sum()
            qtd_div    = (df["status"] != "✅ OK").sum()
            fat_total  = df["desconto_fatura"].sum()
            ref_total  = df["desconto_esperado"].sum()
            diferenca  = fat_total - ref_total
            cor_div    = "#c62828" if qtd_div > 0 else "#2e7d32"
            cor_diff   = "#c62828" if abs(diferenca) > TOLERANCIA else "#2e7d32"
            linhas_resumo.append(
                f"<tr>"
                f"<td style='padding:7px 14px;'>{label}</td>"
                f"<td style='padding:7px 14px; text-align:center;'>{total}</td>"
                f"<td style='padding:7px 14px; text-align:center; color:#2e7d32; font-weight:600;'>{qtd_ok}</td>"
                f"<td style='padding:7px 14px; text-align:center; color:{cor_div}; font-weight:600;'>{qtd_div}</td>"
                f"<td style='padding:7px 14px; text-align:right;'>R$ {fat_total:.2f}</td>"
                f"<td style='padding:7px 14px; text-align:right;'>R$ {ref_total:.2f}</td>"
                f"<td style='padding:7px 14px; text-align:right; font-weight:700; color:{cor_diff};'>R$ {diferenca:+.2f}</td>"
                f"</tr>"
            )

        th = "<th style='background:#1565C0; color:white; font-size:13px; padding:8px 14px; text-align:{a};'>{t}</th>"
        header_resumo = (
            th.format(a="left",   t="Fornecedor") +
            th.format(a="center", t="Colaboradores") +
            th.format(a="center", t="✅ OK") +
            th.format(a="center", t="⚠️ Divergências") +
            th.format(a="right",  t="Fatura (R$)") +
            th.format(a="right",  t="Esperado (R$)") +
            th.format(a="right",  t="Diferença (R$)")
        )

        display(HTML(f"""
        <div style="font-family:sans-serif; margin-bottom:24px;">
          <h3 style="margin:0 0 12px 0; color:#1a1a2e; font-size:16px;">
            📊 Resumo da Conciliação
            <span style="font-weight:normal; color:#555; font-size:13px; margin-left:8px;">{hoje}</span>
          </h3>
          <table style="border-collapse:collapse; font-size:13px; width:100%;">
            <thead><tr>{header_resumo}</tr></thead>
            <tbody>{''.join(linhas_resumo)}</tbody>
          </table>
        </div>
        """))

        # ── Detalhe de divergências por fornecedor ────────────────────────────
        for chave, df in resultados.items():
            label = " ".join(chave).title()
            df_div = df[df["status"] != "✅ OK"].copy()

            if df_div.empty:
                display(HTML(
                    f"<div style='font-family:sans-serif; font-size:14px; margin-bottom:16px; "
                    f"padding:10px 14px; background:#e8f5e9; border-left:3px solid #2e7d32; border-radius:4px;'>"
                    f"✅ <strong>{label}</strong> — sem divergências."
                    f"</div>"
                ))
                continue

            display(HTML(
                f"<h4 style='font-family:sans-serif; font-size:14px; color:#1a1a2e; "
                f"margin:16px 0 8px 0;'>⚠️ Divergências — {label}</h4>"
            ))

            df_exibir = df_div[[
                "matricula", "nome", "qtd_vidas",
                "desconto_esperado", "desconto_fatura", "diferenca", "status"
            ]].copy()
            df_exibir.columns = [
                "Matrícula", "Nome", "Vidas",
                "Esperado (R$)", "Fatura (R$)", "Diferença (R$)", "Status"
            ]

            def _cor_linha(row):
                return ["background-color:#fff8e1; color:#222; font-size:13px;"] * len(row)

            display(
                df_exibir.style
                .apply(_cor_linha, axis=1)
                .format({"Esperado (R$)": "{:.2f}", "Fatura (R$)": "{:.2f}", "Diferença (R$)": "{:+.2f}"})
                .set_table_styles([{
                    "selector": "th",
                    "props": [("background-color", "#1565C0"), ("color", "white"),
                              ("font-size", "13px"), ("padding", "8px 12px"),
                              ("text-align", "left")]
                }])
                .set_properties(**{"padding": "6px 12px"})
                .hide(axis="index")
            )

        # ── Export opcional ───────────────────────────────────────────────────
        if exportar_excel:
            from datetime import datetime as dt
            mes_ano = dt.today().strftime("%m%Y")
            frames = []
            for chave, df in resultados.items():
                df_exp = df.copy()
                df_exp.insert(0, "fornecedor", " ".join(chave).title())
                frames.append(df_exp)
            df_export = pd.concat(frames, ignore_index=True)
            nome_export = f"conciliacao_{mes_ano}.xlsx"
            df_export.to_excel(nome_export, index=False)
            files.download(nome_export)
            display(HTML(f"<p style='font-family:sans-serif; font-size:13px; color:#1565C0;'>📥 Exportado: <strong>{nome_export}</strong></p>"))
